In [1]:
import eikon as ek
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
 
# Set your Eikon API key
ek.set_app_key('7124cc602cee484ab6297f9468d491c9f585afdd')
 
ric1 = 'NGZ25'
ric2 = 'NGM26'
 
# Point value
POINT_VALUE = 1000  # ₹1000 per point
 
# Download data
prices_1m = ek.get_timeseries(
    [ric1, ric2],
    start_date='2025-05-01',
    end_date='2025-08-08',
    interval='minute'
).dropna()
 
# Handle MultiIndex structure
if isinstance(prices_1m.columns, pd.MultiIndex):
    data_1m = pd.concat([
        prices_1m[(ric1, 'OPEN')].rename('open_1'),
        prices_1m[(ric1, 'HIGH')].rename('high_1'),
        prices_1m[(ric1, 'LOW')].rename('low_1'),
        prices_1m[(ric1, 'CLOSE')].rename('close_1'),
        prices_1m[(ric1, 'VOLUME')].rename('volume_1'),
        prices_1m[(ric2, 'OPEN')].rename('open_2'),
        prices_1m[(ric2, 'HIGH')].rename('high_2'),
        prices_1m[(ric2, 'LOW')].rename('low_2'),
        prices_1m[(ric2, 'CLOSE')].rename('close_2'),
        prices_1m[(ric2, 'VOLUME')].rename('volume_2'),
    ], axis=1).astype(float)
else:
    raise ValueError("Unexpected columns structure; expected MultiIndex columns for OHLCV")
 
# Resample to 30-minute
resample_dict = {
    'open_1': 'first',
    'high_1': 'max',
    'low_1': 'min',
    'close_1': 'last',
    'volume_1': 'sum',
    'open_2': 'first',
    'high_2': 'max',
    'low_2': 'min',
    'close_2': 'last',
    'volume_2': 'sum',
}
data_15m = data_1m.resample('15T').agg(resample_dict).dropna()
 
# Close prices
y0 = data_15m['close_1']
y1 = data_15m['close_2']
 
# Cointegration test
score, pvalue, _ = coint(y0, y1)
print(f"Cointegration test p-value: {pvalue}")
 
if pvalue < 0.1:
    x = sm.add_constant(y1)
    model = sm.OLS(y0, x)
    result = model.fit()
    hedge_ratio = result.params[y1.name]
    print(f"Hedge ratio: {hedge_ratio}")
 
    # Spread and Z-score
    spread = y0 - hedge_ratio * y1
    window = 30
    spread_mean = spread.rolling(window=window).mean()
    spread_std = spread.rolling(window=window).std()
    zscore = (spread - spread_mean) / spread_std
 
    bktest = pd.DataFrame({
        'y0': y0,
        'y1': y1,
        'spread': spread,
        'zscore': zscore
    })
 
    entry_threshold = 2
    stop_loss_threshold = 4
    target_threshold = 0
 
    bktest['position'] = 0
    position = 0
    trade_log = []
    trade = None
 
    for i in range(1, len(bktest)):
        z = bktest['zscore'].iloc[i]
        idx = bktest.index[i]
 
        if position == 0:
            if z < -entry_threshold:
                position = 1
                trade = {
                    'entry_time': idx,
                    'entry_z': z,
                    'type': 'long',
                    'entry_y0': bktest['y0'].iloc[i],
                    'entry_y1': bktest['y1'].iloc[i],
                    'entry_spread': bktest['spread'].iloc[i]
                }
            elif z > entry_threshold:
                position = -1
                trade = {
                    'entry_time': idx,
                    'entry_z': z,
                    'type': 'short',
                    'entry_y0': bktest['y0'].iloc[i],
                    'entry_y1': bktest['y1'].iloc[i],
                    'entry_spread': bktest['spread'].iloc[i]
                }
 
        elif position == 1:
            if z >= target_threshold or z <= -stop_loss_threshold:
                pnl = (
                    (bktest['y0'].iloc[i] - trade['entry_y0']) -
                    hedge_ratio * (bktest['y1'].iloc[i] - trade['entry_y1'])
                ) * POINT_VALUE
 
                trade.update({
                    'exit_time': idx,
                    'exit_z': z,
                    'exit_y0': bktest['y0'].iloc[i],
                    'exit_y1': bktest['y1'].iloc[i],
                    'exit_spread': bktest['spread'].iloc[i],
                    'PnL': pnl,
                    'duration': (idx - trade['entry_time'])
                })
                trade_log.append(trade)
                trade = None
                position = 0
 
        elif position == -1:
            if z <= target_threshold or z >= stop_loss_threshold:
                pnl = (
                    -(bktest['y0'].iloc[i] - trade['entry_y0']) +
                    hedge_ratio * (bktest['y1'].iloc[i] - trade['entry_y1'])
                ) * POINT_VALUE
 
                trade.update({
                    'exit_time': idx,
                    'exit_z': z,
                    'exit_y0': bktest['y0'].iloc[i],
                    'exit_y1': bktest['y1'].iloc[i],
                    'exit_spread': bktest['spread'].iloc[i],
                    'PnL': pnl,
                    'duration': (idx - trade['entry_time'])
                })
                trade_log.append(trade)
                trade = None
                position = 0
 
        bktest.loc[idx, 'position'] = position
 
    # Calculate PnL
    bktest['pnl'] = (
        bktest['position'].shift(1) *
        (bktest['y0'].diff() - hedge_ratio * bktest['y1'].diff())
    ) * POINT_VALUE
 
    bktest['cum_pnl'] = bktest['pnl'].cumsum()
 
    total_pnl = bktest['cum_pnl'].iloc[-1]
    n_trades = ((bktest['position'].diff().abs() > 0).sum()) // 2
    gross_profit = bktest['pnl'][bktest['pnl'] > 0].sum()
    gross_loss = bktest['pnl'][bktest['pnl'] < 0].sum()
    trades = pd.DataFrame(trade_log)
    win_rate = (trades['PnL'] > 0).mean()
    max_drawdown = (bktest['cum_pnl'].cummax() - bktest['cum_pnl']).max()
 
    print(f"\nPOINT VALUE: ₹{POINT_VALUE}")
    print(f"TOTAL PnL: ₹{total_pnl:,.2f}")
    print(f"NUMBER OF TRADES: {n_trades}")
    print(f"GROSS PROFIT: ₹{gross_profit:,.2f}")
    print(f"GROSS LOSS: ₹{gross_loss:,.2f}")
    print(f"WIN RATE: {win_rate:.2%}")
    print(f"MAX DRAWDOWN: ₹{max_drawdown:,.2f}")
 
    # Trade log
    trade_log_df = pd.DataFrame(trade_log)
    pd.set_option('display.max_rows', None)
    print("\nTrade Log:")
    print(trade_log_df[['entry_time', 'exit_time', 'type', 'entry_z', 'exit_z', 'PnL', 'duration']])
 
    # Plot results
    plt.figure(figsize=(14, 7))
    plt.subplot(2, 1, 1)
    plt.plot(bktest['cum_pnl'], label='Cumulative PnL (₹)')
    plt.title('Pairs Trading Backtest with ₹1000/pt')
    plt.legend()
    plt.subplot(2, 1, 2)
    plt.plot(bktest['zscore'], label='Spread Z-score')
    plt.plot(bktest['position'] * 2, label='Position (scaled)')
    plt.axhline(entry_threshold, color='red', linestyle='--', label='Entry Threshold')
    plt.axhline(-entry_threshold, color='green', linestyle='--')
    plt.axhline(stop_loss_threshold, color='darkred', linestyle=':', label='Stop Loss')
    plt.axhline(-stop_loss_threshold, color='darkgreen', linestyle=':')
    plt.axhline(target_threshold, color='black', linestyle='-', label='Target')
    plt.legend()
    plt.tight_layout()
    plt.show()
 
else:
    print("Selected pair is not cointegrated. Try another.")
# Export data to Excel
output_filename = "pairs_trading_backtest_output.xlsx"
with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    data_15m.to_excel(writer, sheet_name='15min_Prices')
    bktest.to_excel(writer, sheet_name='Backtest')
    trade_log_df.to_excel(writer, sheet_name='Trade_Log')
 
print(f"\n✅ Exported results to: {output_filename}")

2025-12-08 18:02:43,762 P[6192] [MainThread 4092] Error: no proxy address identified.
Check if Eikon Desktop or Eikon API Proxy is running.
2025-12-08 18:02:43,771 P[6192] [MainThread 4092] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2025-12-08 18:02:43,776 P[6192] [MainThread 4092] Error on handshake url http://127.0.0.1:None/api/handshake : InvalidURL("Invalid port: 'None'")
2025-12-08 18:02:43,776 P[6192] [MainThread 4092] Port number was not identified, cannot send any request
2025-12-08 18:02:43,791 P[6192] [MainThread 4092] HTTP request failed: Invalid port: 'None'


AttributeError: 'NoneType' object has no attribute 'dropna'